# NBT modelling summary

This notebook compares the winning data treatment, feature configuration and algorithm across the three prespecified prediction targets. Model selection occurred in development cross-validation; the values below are from the frozen untouched test sets.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULT_DIR = PROJECT_ROOT / "result" / "modeling"
winner_summary = pd.read_csv(RESULT_DIR / "cross_target_winner_summary.csv")
winner_summary

,target,winning missing strategy,winning feature configuration,winning model,primary metric,primary value,secondary metric,secondary value
0,duration_error_mins,"Missing-aware, priority retained",Both procedure levels,XGBoost,MAE,30.881546,R2,0.435762
1,operation_length_mins,"Missing-aware, priority excluded",Both procedure levels,XGBoost,MAE,30.295510,R2,0.717485
2,meaningful_overrun_flag,"Missing-aware, priority retained",Both procedure levels,XGBoost,PR-AUC,0.795178,Recall,0.782037


## Winning type of data

The winner table states whether priority was retained through the selected missing strategy, which feature representation won, and whether the neural network or another algorithm performed best. Start hour, flagged-record exclusion and complete cases remain sensitivity analyses rather than primary data choices.

In [2]:
importance = pd.read_csv(RESULT_DIR / "cross_target_feature_importance.csv")
top_features = (
    importance.sort_values(["target", "rank"])
    .groupby("target", as_index=False, group_keys=False)
    .head(10)
)
top_features

,target,feature,importance mean,importance SD,rank
0,duration_error_mins,ExpectedDurationMins,15.066025,0.342264,1
1,duration_error_mins,anaesthetic_desc,4.506346,0.329378,2
2,duration_error_mins,procedure_code_group,3.951615,0.194191,3
3,duration_error_mins,procedure_code_category,3.707276,0.296642,4
4,duration_error_mins,intended_management_label,3.137609,0.164329,5
5,duration_error_mins,admission_type_label,2.577315,0.184294,6
6,duration_error_mins,ASAScore,0.768784,0.089858,7
7,duration_error_mins,age_at_operation,0.456571,0.079252,8
8,duration_error_mins,sex_national_code,0.194826,0.062420,9
9,duration_error_mins,priority_level_label,-0.002235,0.100916,10


## Interpretation

Permutation importance measures predictive contribution on the frozen test population and is not a causal effect. Correlated features can share importance. A low-ranked feature may still be clinically important, and external validation remains necessary before deployment.